# 02 – DuckDB

> Runs against the catalog from `CATALOG` (Lakekeeper, Polaris, RustFS or Unity; default Lakekeeper). What a catalog cannot do is shown by the cell instead of aborting.

DuckDB attaches the catalog with `ATTACH` and can read **and** write
(`INSERT`, `UPDATE`, `DELETE`, also v3 tables).

- Warehouse with vended credentials (Lakekeeper `demo-sts`, Polaris, Unity): works directly with the catalog's credentials.
- Remote signing (Lakekeeper only, `demo-signing`): DuckDB **cannot do remote signing** – static keys are needed there (`CREATE SECRET`).
- RustFS catalog: DuckDB cannot attach it (it derives the SigV4 service from an AWS host name) – the notebook ends there with this known gap.

In [ ]:
from iceberg_lab.config import PROFILE, REMOTE_SIGNING, SETTINGS, WAREHOUSE_SIGNING, WAREHOUSE_STS
from iceberg_lab.notebook import catalog_limit, header

CATALOG_URI = SETTINGS.catalog_uri
S3_ENDPOINT = SETTINGS.s3_endpoint
header()

In [ ]:
from iceberg_lab.engines import DuckDBEngine

duck = DuckDBEngine(WAREHOUSE_STS)  # attached as "lk", with the catalog's login
con = duck.con
con.sql("SHOW ALL TABLES").df()[["database", "schema", "name"]]

## Create a namespace and a table, write

In [ ]:
con.sql("CREATE SCHEMA IF NOT EXISTS lk.nb_duckdb")
con.sql("DROP TABLE IF EXISTS lk.nb_duckdb.sensors")
con.sql("""
    CREATE TABLE lk.nb_duckdb.sensors (
        sensor_id INTEGER, measured_at TIMESTAMP, temperature DOUBLE, tags VARCHAR[]
    )""")
con.sql("""
    INSERT INTO lk.nb_duckdb.sensors
    SELECT i % 5, TIMESTAMP '2026-11-01' + INTERVAL (i) MINUTE, 15 + random() * 10,
           CASE WHEN i % 2 = 0 THEN ['even'] ELSE ['odd', 'x'] END
    FROM range(1000) t(i)""")
con.sql(
    "SELECT sensor_id, count(*), round(avg(temperature), 2) FROM lk.nb_duckdb.sensors GROUP BY ALL ORDER BY 1"
).df()

## UPDATE and DELETE (row level)

In [ ]:
con.sql(
    "UPDATE lk.nb_duckdb.sensors SET temperature = -999 WHERE sensor_id = 0 AND measured_at < TIMESTAMP '2026-11-01 00:10:00'"
)
con.sql("DELETE FROM lk.nb_duckdb.sensors WHERE sensor_id = 4")
con.sql(
    "SELECT sensor_id, count(*), min(temperature) FROM lk.nb_duckdb.sensors GROUP BY ALL ORDER BY 1"
).df()

## Snapshots and time travel

In [ ]:
snaps = con.sql(
    "SELECT * FROM iceberg_snapshots(lk.nb_duckdb.sensors) ORDER BY sequence_number"
).df()
snaps[["sequence_number", "snapshot_id", "timestamp_ms"]]

In [ ]:
first = int(snaps.snapshot_id.iloc[0])
con.sql(f"SELECT count(*) AS rows_then FROM lk.nb_duckdb.sensors AT (VERSION => {first})").df()

## Create a v3 table

In [ ]:
with catalog_limit("v3 table with TIMESTAMP_NS"):
    con.sql("DROP TABLE IF EXISTS lk.nb_duckdb.events_v3")
    con.sql(
        "CREATE TABLE lk.nb_duckdb.events_v3 (id BIGINT, ts TIMESTAMP_NS) WITH ('format-version' = '3')"
    )
    con.sql(
        "INSERT INTO lk.nb_duckdb.events_v3 VALUES (1, TIMESTAMP_NS '2026-11-13 12:00:00.123456789')"
    )
    print(con.sql("SELECT * FROM lk.nb_duckdb.events_v3").df())

## Warehouse with remote signing: only with static keys

In [ ]:
if not REMOTE_SIGNING:
    print(f"Catalog {PROFILE.name} offers no remote signing - one warehouse only")
else:
    con.sql(f"""
        CREATE OR REPLACE SECRET rustfs (
            TYPE s3, KEY_ID '{SETTINGS.s3_access_key}', SECRET '{SETTINGS.s3_secret_key}',
            ENDPOINT '{SETTINGS.s3_host_port}', URL_STYLE 'path', USE_SSL false, REGION '{SETTINGS.s3_region}'
        )""")
    con.sql(f"""
        ATTACH '{WAREHOUSE_SIGNING}' AS lk_signing (
            TYPE iceberg, ENDPOINT '{CATALOG_URI}', AUTHORIZATION_TYPE 'none',
            ACCESS_DELEGATION_MODE 'none'
        )""")
    con.sql("CREATE SCHEMA IF NOT EXISTS lk_signing.nb_duckdb")
    # DuckDB's Iceberg extension has no CREATE OR REPLACE -> DROP + CREATE
    con.sql("DROP TABLE IF EXISTS lk_signing.nb_duckdb.copy")
    con.sql("CREATE TABLE lk_signing.nb_duckdb.copy AS SELECT * FROM lk.nb_duckdb.sensors LIMIT 10")
    print(con.sql("SELECT count(*) FROM lk_signing.nb_duckdb.copy").df())